# Cohere Rerank Models in SageMaker  


In this notebook, we demonstrate how to use the SageMaker Python SDK to deploy and run inference on Rerank models from Cohere.

## Setup



First, upgrade to the latest Sagemaker, Boto and Cohere SDKs to be able to deploy and invoke Cohere models.

In [ ]:
%pip install --upgrade --quiet sagemaker boto3 cohere

## Deploy

#### Latest Models
| **Model Name**        | **Model ID**                       |
|----------------------|------------------------------------|
| Rerank 5 Pro | `cohere-rerank-v5-pro` |
| Rerank 5 Fast | `cohere-rerank-v5-fast` |

#### Older Models

| **Model Name**        | **Model ID**                       |
|----------------------|------------------------------------|
| Rerank v4 Pro | `cohere-rerank-v4-pro` |
| Rerank v4 Fast | `cohere-rerank-v4-fast` |
| Rerank v3.5 | `cohere-rerank-v3.5` |
| Rerank v3 Multilingual | `cohere-rerank-v3-multilingual` |
| Rerank v3 English | `cohere-rerank-v3-english`      |
| Rerank "Nimble" v3 Multilingual | `cohere-rerank-nimble-multi` |
| Rerank "Nimble" v3 English | `cohere-rerank-nimble-english`      |
| Rerank v2 Multilingual | `cohere-rerank-multilingual-v2`      |
| Rerank v2 English | `cohere-rerank-english-v2`      |

Specify the model ID to deploy one of the models from the above list.

In [ ]:
model_id, model_version = input("Please enter a model ID"), "*"

Use the Sagemaker Jumpstart SDK to deploy the model.

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

model = JumpStartModel(model_id=model_id, model_version=model_version)
deployed_model = model.deploy()

# Use the model

In [ ]:
import cohere

# Initialize via parameters or local env vars / config file
co = cohere.SagemakerClientV2()

docs = [
    "Carson City is the capital city of the American state of Nevada.",
    "The Commonwealth of the Northern Mariana Islands is a group of islands in the Pacific Ocean. Its capital is Saipan.",
    "Capitalization or capitalisation in English grammar is the use of a capital letter at the start of a word. English usage varies from capitalization in other languages.",
    "Washington, D.C. (also known as simply Washington or D.C., and officially as the District of Columbia) is the capital of the United States. It is a federal district.",
    "Capital punishment has existed in the United States since beforethe United States was a country. As of 2017, capital punishment is legal in 30 of the 50 states.",
]

response = co.rerank(
    model=deployed_model.endpoint_name,
    query="What is the capital of the United States?",
    documents=docs,
    top_n=3,
)

print(response.results)

### Rerank 5 with Context Focus

Rerank 5 (`rerank-v5.0-pro` and `rerank-v5.0-fast`) adds **Context Focus**, which returns the most relevant spans within each document in the same Rerank call. Instead of sending whole documents to your LLM, you can send only the matching passages — cutting token costs — or highlight them in your interface to show users why a result matched.

Context Focus is off by default. To enable it, pass `relevant_spans` in the request:

- `mode`: set to `"enabled"` to turn on Context Focus
- `threshold`: minimum span score (0–1). A higher value removes more text
- `max_spans`: caps the number of spans returned per document

Each result includes its calibrated `relevance_score` and, where passages meet the `threshold`, a `relevant_spans` list with the `start` and `end` character offsets of each passage in the original document and its `score`. If no passage meets the threshold, `relevant_spans` is omitted for that result.

In [ ]:
# Rerank 5 with Context Focus enabled.

long_docs = [
    "Carson City is the capital city of the American state of Nevada. At the 2010 United States Census, Carson City had a population of 55,274. The city was founded in 1858 and is named after the mountain man Kit Carson.",
    "The Commonwealth of the Northern Mariana Islands is a group of islands in the Pacific Ocean that are a political division controlled by the United States. Its capital is Saipan. The islands were settled around 1500 BC.",
    "Washington, D.C. (also known as simply Washington or D.C., and officially as the District of Columbia) is the capital of the United States. It is a federal district. The President of the USA and many major national government offices are in the territory. This makes it the political center of the United States of America.",
    "Capital punishment has existed in the United States since before the United States was a country. As of 2017, capital punishment is legal in 30 of the 50 states. The federal government also employs it.",
]

response = co.rerank(
    model=deployed_model.endpoint_name,  # endpoint of a deployed Rerank 5 model
    query="What is the capital of the United States?",
    documents=long_docs,
    top_n=3,
    relevant_spans={"mode": "enabled", "threshold": 0.3, "max_spans": 10},
)

for result in response.results:
    print(f"index={result.index} relevance_score={result.relevance_score:.4f}")
    for span in getattr(result, "relevant_spans", None) or []:
        passage = long_docs[result.index][span.start : span.end]
        print(f"  span score={span.score:.4f} [{span.start}:{span.end}] -> {passage!r}")

### More API Features
To learn more about API features please see the documentation in the [Cohere Rerank API Reference](https://docs.cohere.com/reference/rerank).

# Clean Up

After using the resource, you can delete the model and the endpoint.

In [ ]:
deployed_model.delete_model()
deployed_model.delete_endpoint()